# 02 — Data Cleaning and Preprocessing

This notebook cleans the flattened SQuAD 2.0 records and constructs a unique passage collection. It preserves original text, answer annotations, question IDs and source metadata. Sampling is intentionally deferred until after article-group splitting.

## 1. Imports and paths

This notebook expects the outputs of Notebook 01 in `data/processed/`. It reloads the original JSON-derived tables so that all answer annotations remain intact.

In [ ]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
PROCESSED = ROOT / "data" / "processed"
RAW = ROOT / "data" / "raw"
for p in [PROCESSED, RAW]: p.mkdir(parents=True, exist_ok=True)

def load_exploration(name):
    df = pd.read_csv(PROCESSED / f"{name}_exploration.csv")
    df["answers"] = df["answers"].apply(json.loads)
    df["is_answerable"] = df["is_answerable"].astype(bool)
    df["is_impossible"] = df["is_impossible"].astype(bool)
    return df

train_df = load_exploration("train")
dev_df = load_exploration("dev")
all_df = pd.concat([train_df, dev_df], ignore_index=True)
print("Combined rows:", len(all_df))
print("Unique question IDs:", all_df.question_id.nunique())

## 2. Audit and filter unusable records

The proposal permits excluding examples with unusable or malformed text. We make the filtering rules explicit and report how many records each rule removes. We do **not** remove examples simply because their context is long or their question is difficult.

In [ ]:
before = len(all_df)
checks = {
    "missing_question": all_df.question.isna(),
    "empty_question": all_df.question.fillna("").str.strip().eq(""),
    "missing_context": all_df.context.isna(),
    "empty_context": all_df.context.fillna("").str.strip().eq(""),
    "missing_question_id": all_df.question_id.isna(),
}
display(pd.DataFrame({"rule": list(checks), "rows_flagged": [int(v.sum()) for v in checks.values()]}))

invalid = np.logical_or.reduce([v.to_numpy() for v in checks.values()])
clean_df = all_df.loc[~invalid].copy()
clean_df = clean_df.drop_duplicates(subset=["question_id"], keep="first").copy()

print(f"Rows before: {before:,}")
print(f"Rows after:  {len(clean_df):,}")
print(f"Removed:     {before-len(clean_df):,}")

## 3. Validate answer annotations

For answerable records, check that at least one supplied answer text occurs at its annotated offset. This is a diagnostic rather than an excuse to silently delete examples. We retain the records and mark offset validity so the impact can be reviewed.

In [ ]:
def has_valid_answer_annotation(row):
    if not row.is_answerable:
        return True
    context = row.context
    for ans in row.answers:
        text = ans.get("text")
        start = ans.get("answer_start")
        if isinstance(text, str) and isinstance(start, int):
            if context[start:start+len(text)] == text:
                return True
    return False

clean_df["answer_annotation_valid"] = clean_df.apply(has_valid_answer_annotation, axis=1)
display(clean_df.groupby("is_answerable")["answer_annotation_valid"].agg(["count", "mean"]))
print("Answerable records with no valid offset:",
      int((clean_df.is_answerable & ~clean_df.answer_annotation_valid).sum()))

## 4. Build a unique passage table

The retriever indexes passages, not question rows. Multiple questions can refer to the same paragraph. Create one row per unique paragraph and preserve article title, paragraph ID, source split and context text.

In [ ]:
passages_df = (
    clean_df[["paragraph_id", "article_title", "source_split", "context"]]
    .drop_duplicates(subset=["paragraph_id"])
    .reset_index(drop=True)
)
passages_df["passage_length_chars"] = passages_df.context.str.len()
print("Question rows:", len(clean_df))
print("Unique passages:", len(passages_df))
display(passages_df.head())

## 5. Text normalization diagnostics

We create normalized helper columns for analysis and lexical matching. We preserve original `question` and `context` unchanged because offsets and answer extraction refer to the original text.

Normalization here is deliberately light: lowercase and whitespace normalization. Do not overwrite source text.

In [ ]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

clean_df["question_normalized"] = clean_df.question.map(normalize_text)
clean_df["context_normalized"] = clean_df.context.map(normalize_text)
passages_df["context_normalized"] = passages_df.context.map(normalize_text)

display(clean_df[["question", "question_normalized", "is_answerable"]].head())

## 6. Inspect class counts before sampling

The target subset is approximately 6,000–10,000 examples, subject to the final inspection. The proposal also allows reducing to 2,000–3,000 if resources or article-level grouping make the larger sample impractical.

We will not sample yet. Group-aware splitting happens first in Notebook 03; sampling before splitting could place questions from the same article in multiple splits.

In [ ]:
display(clean_df["is_answerable"].value_counts().rename(index={True:"Answerable", False:"Unanswerable"}).to_frame("count"))
print("Unique articles:", clean_df.article_title.nunique())
print("Unique passages:", clean_df.paragraph_id.nunique())
print("Max questions per passage:", clean_df.groupby("paragraph_id").size().max())

## 7. Save cleaned question and passage tables

We save the complete cleaned pool. The balanced subset and group-based splits are created in the next notebook. JSONL is used for nested answer annotations; CSV is also saved for convenient inspection.

In [ ]:
# Store nested answer annotations as JSON strings in CSV.
export_questions = clean_df.copy()
export_questions["answers"] = export_questions["answers"].apply(
    lambda x: json.dumps(x, ensure_ascii=False)
)
export_questions.to_csv(PROCESSED / "questions_clean.csv", index=False, encoding="utf-8")
passages_df.to_csv(PROCESSED / "passages_clean.csv", index=False, encoding="utf-8")

# JSONL preserves nested structures naturally.
with (PROCESSED / "questions_clean.jsonl").open("w", encoding="utf-8") as f:
    for row in clean_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

print("Saved cleaned question and passage tables to", PROCESSED)

## 8. Review questions

1. Why should a retriever index unique passages rather than one copy per question?
2. Why do we preserve original text alongside normalized text?
3. Why should we split by article before balanced sampling?
4. Which filtering rules did we apply, and how many examples did each flag?

**Next:** Notebook 03 creates reproducible article-group train/validation/test splits and a balanced subset.